## Imports

In [7]:
from pptx import Presentation
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import os

##  Function to extract text from any PPTX

In [8]:
def extract_slide_texts(pptx_path):
    prs = Presentation(pptx_path)
    slide_texts = []
    for slide in prs.slides:
        text = ""
        for shape in slide.shapes:
            if shape.has_text_frame:
                for para in shape.text_frame.paragraphs:
                    text += para.text + " "
        slide_texts.append(text.strip())
    return slide_texts

##  Function to generate consistency report

In [9]:
def consistency_report(pptx_path, threshold=0.3):
    print(f"\n{'='*50}")
    print(f"FILE: {os.path.basename(pptx_path)}")
    print(f"{'='*50}")

    # Extract texts
    slide_texts = extract_slide_texts(pptx_path)

    # Generate embeddings
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(slide_texts)

    # Compute similarity
    similarity_matrix = cosine_similarity(embeddings)

    # Report
    all_consistent = True
    for i in range(len(slide_texts) - 1):
        score = similarity_matrix[i][i+1]
        status = "✅ OK" if score >= threshold else "⚠️  INCONSISTENT"
        print(f"Slide {i+1} vs Slide {i+2}: Score={score:.2f}  {status}")
        if score < threshold:
            all_consistent = False

    if all_consistent:
        print("\n✅ Overall: All slides are consistent!")
    else:
        print("\n⚠️  Overall: Inconsistencies detected!")

##  Run on all 3 PPTX files

In [10]:
test_folder = "test_slides"

for filename in os.listdir(test_folder):
    if filename.endswith(".pptx"):
        full_path = os.path.join(test_folder, filename)
        consistency_report(full_path, threshold=0.3)


FILE: consistent.pptx


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Slide 1 vs Slide 2: Score=0.72  ✅ OK
Slide 2 vs Slide 3: Score=0.59  ✅ OK
Slide 3 vs Slide 4: Score=0.50  ✅ OK
Slide 4 vs Slide 5: Score=0.45  ✅ OK
Slide 5 vs Slide 6: Score=0.52  ✅ OK

✅ Overall: All slides are consistent!

FILE: inconsistent.pptx


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Slide 1 vs Slide 2: Score=0.30  ⚠️  INCONSISTENT
Slide 2 vs Slide 3: Score=0.09  ⚠️  INCONSISTENT
Slide 3 vs Slide 4: Score=-0.02  ⚠️  INCONSISTENT
Slide 4 vs Slide 5: Score=0.08  ⚠️  INCONSISTENT
Slide 5 vs Slide 6: Score=0.10  ⚠️  INCONSISTENT

⚠️  Overall: Inconsistencies detected!

FILE: partial.pptx


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Slide 1 vs Slide 2: Score=0.68  ✅ OK
Slide 2 vs Slide 3: Score=0.24  ⚠️  INCONSISTENT
Slide 3 vs Slide 4: Score=0.06  ⚠️  INCONSISTENT
Slide 4 vs Slide 5: Score=0.16  ⚠️  INCONSISTENT
Slide 5 vs Slide 6: Score=0.15  ⚠️  INCONSISTENT

⚠️  Overall: Inconsistencies detected!


### Test different thresholds

In [15]:
thresholds = [0.1, 0.2, 0.3, 0.36, 0.4, 0.5]

test_folder = "test_slides"
files = [f for f in os.listdir(test_folder) if f.endswith(".pptx")]

print(f"{'Threshold':<12}", end="")
for f in files:
    print(f"{f:<25}", end="")
print()
print("-" * 70)

for thresh in thresholds:
    print(f"{thresh:<12}", end="")
    for filename in files:
        full_path = os.path.join(test_folder, filename)
        slide_texts = extract_slide_texts(full_path)
        model = SentenceTransformer("all-MiniLM-L6-v2")
        embeddings = model.encode(slide_texts)
        similarity_matrix = cosine_similarity(embeddings)
        
        flagged = 0
        for i in range(len(slide_texts) - 1):
            score = similarity_matrix[i][i+1]
            if score < thresh:
                flagged += 1
        
        print(f"{flagged} slides flagged      ", end="")
    print()

Threshold   consistent.pptx          inconsistent.pptx        partial.pptx             
----------------------------------------------------------------------
0.1         

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0 slides flagged      

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

4 slides flagged      

...

1 slides flagged      

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

5 slides flagged      

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

4 slides flagged      
